# Construcci?n de Pipeline del Modelo

## Importaciones

In [ ]:
# Manejo de datos y operaciones
import numpy as np
import pandas as pd

# Visualizaci?n
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento de datos
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE


# Modelos de machine learning
from sklearn.ensemble import RandomForestClassifier

# M?tricas
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Manejo de archivos
import os
from pathlib import Path

# Miscellaneous
import warnings
warnings.filterwarnings('ignore') 



## Lectura de los Datos

In [ ]:
#Cargar los datos despues  

df = pd.read_csv("../stage/train.csv")

In [ ]:
# Separar las caracter?sticas (X) y la variable objetivo (y)
X = df.drop(columns=['target'])  # Excluir la columna target
y = df['target']  # Variable objetivo

# Dividir el conjunto en train + validation (80%) y test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Dividir el conjunto de train en train (80% de train) y validation (20% de train)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

In [ ]:
smote = SMOTE(random_state=42)

# --- Random Forest (sin escalado, pero con SMOTE) ---
x_rf_train, y_rf_train = smote.fit_resample(X_train, y_train)
x_rf_val, y_rf_val = X_val, y_val  # Sin escalado
x_rf_test, y_rf_test = X_test, y_test  # Sin escalado

## Construcci?n del Pipeline

In [ ]:
class LabelEncoderWrapper(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
    
    def fit(self, X, y=None):
        for col in X.columns:
            self.encoders[col] = LabelEncoder()
            self.encoders[col].fit(X[col])
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        for col in X.columns:
            X_copy[col] = self.encoders[col].transform(X[col])
        return X_copy

In [ ]:
# Identificar columnas categ?ricas y num?ricas
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(exclude=['object']).columns

# Paso 1: Transformador para columnas categ?ricas
categorical_transformer = Pipeline(steps=[
    ('label_encoder', LabelEncoderWrapper())  # Wrapper para usar LabelEncoder en pipelines
])

# Paso 2: Combinaci?n de transformadores con ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'  # Las columnas num?ricas se quedan igual
)

# Paso 3: Definir el pipeline completo
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Entrenar el pipeline
pipeline.fit(x_rf_train, y_rf_train)

# Predecir y evaluar
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Se procede a guardar el segundo modelo para su despliegue posterior
joblib.dump(pipeline, '../../deploy/rf_pipeline.joblib')